In [1]:
from dataclasses import dataclass
from enum import Enum, auto
class Action(Enum):
    F = auto()
    B = auto()


@dataclass
class MicroBatch:
    action:Action
    mb_id:int
    prev_node:int
    next_node:int

    def __repr__(self):
        return f"<action : {self.action.name}, id : {self.mb_id}, p : {self.prev_node}, n : {self.next_node}>\n"
    def __iter__(self):
        yield self.action
        yield self.mb_id
        



class MicroBatchSchedular:
    def __init__(
        self, 
        world_size:int, 
        world_rank:int,
        num_microbatches:int = 8,
        max_activation_size:int = 4,

        ):
        self.world_size = world_size
        self.world_rank = world_rank
        self.num_microbatches = num_microbatches
        self.max_activation_size = max_activation_size
        if num_microbatches < max_activation_size:
            raise AttributeError(f"num_microbatches({num_microbatches}) can not be lesser than max_activation_size({max_activation_size})")
        
    def build(self):
        schedulers = list()
        forward_lists = list()
        backward_lists = list()



        for i in range(self.world_size):
            schedulers.append(list())
            forward_lists.append(list())
            backward_lists.append(list())

        forward_lists[0] = list(range(self.num_microbatches))

        backward_round = [False] * self.world_size
        activation_count = [0] * self.world_size
        while True:
            no_action = 0
            for i in range(self.world_size):
                if backward_round[i]:
                    if len(backward_lists[i]) != 0:
                        back_id = backward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.B,back_id, i-1 if i != 0 else None, i+1 if i != self.world_size-1 else None))
                        activation_count[i] -= 1
                        if i != 0:
                            backward_lists[i-1].append(back_id)
                        if len(forward_lists[i]) != 0:
                            backward_round[i] = False
                    elif len(forward_lists[i]) != 0: #not likely to come in.
                        backward_round[i] = False
                    else:
                        no_action += 1

                else:
                    if len(forward_lists[i]) != 0 and activation_count[i] < self.max_activation_size:
                        for_id = forward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.F, for_id, i-1 if i != 0 else None, i+1 if i != self.world_size-1 else None))
                        activation_count[i] += 1
                        if i == self.world_size - 1:
                            backward_lists[i].append(for_id)
                        else:
                            forward_lists[i+1].append(for_id)
                        if len(backward_lists[i]) != 0:
                            backward_round[i] = True
                    elif len(backward_lists[i]) != 0:
                        back_id = backward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.B, back_id, i-1 if i != 0 else None, i+1 if i != self.world_size-1 else None))
                        activation_count[i] -= 1
                        if i != 0:
                            backward_lists[i-1].append(back_id)
                        
                    else:
                        no_action += 1
            # print(activation_count)
            if no_action == self.world_size:
                break
        return schedulers

In [2]:
schedules = MicroBatchSchedular(4, 0).build()

In [6]:
print(schedules[3])

[<action : F, id : 0, p : 2, n : None>
, <action : B, id : 0, p : 2, n : None>
, <action : F, id : 1, p : 2, n : None>
, <action : B, id : 1, p : 2, n : None>
, <action : F, id : 2, p : 2, n : None>
, <action : B, id : 2, p : 2, n : None>
, <action : F, id : 3, p : 2, n : None>
, <action : B, id : 3, p : 2, n : None>
, <action : F, id : 4, p : 2, n : None>
, <action : B, id : 4, p : 2, n : None>
, <action : F, id : 5, p : 2, n : None>
, <action : B, id : 5, p : 2, n : None>
, <action : F, id : 6, p : 2, n : None>
, <action : B, id : 6, p : 2, n : None>
, <action : F, id : 7, p : 2, n : None>
, <action : B, id : 7, p : 2, n : None>
]


In [58]:
class Node:
    def __init__(self, node_num):
        self.node_num = node_num
        self.ctxs = {}

    def forward(self, mb_id, X):
        print("forward", mb_id)
        self.ctxs[mb_id] = X
    def backward(self, mb_id, output_grad):
        print("backward", mb_id)
        #backward(self.ctxs[mb_id], output_grad)
        return None
    
    def f_recv(self, mb_id):
        print("f_recv", mb_id)
        
    def f_send(self, mb_id, data):
        print("f_send", mb_id)
        
    
    def b_recv(self, mb_id):
        print("b_recv", mb_id)

    def b_send(self, mb_id, data):
        print("b_send", mb_id)


    

In [62]:
class Operator:
    def __init__(self, node_num):
        self.pipe_layer = Node(node_num)
        self.schedules = MicroBatchSchedular(4, 4)
        self.node_num = node_num
    def run(self):
        while True:
            schedule = self.schedules.build()[self.node_num]
            for op, mb_id in schedule:
                if op == Action.F:
                    self.pipe_layer.f_recv(mb_id)
                    self.pipe_layer.forward(mb_id, None)
                    self.pipe_layer.f_send(mb_id, None)
                elif op == Action.B:
                    self.pipe_layer.b_recv(mb_id)
                    self.pipe_layer.backward(mb_id, None)
                    self.pipe_layer.b_send(mb_id, None)
                else:
                    raise Exception(f"지랄")
            break
            
        


In [63]:
op = Operator(0)

In [64]:
op.run()

f_recv 0
forward 0
f_send 0
f_recv 1
forward 1
f_send 1
f_recv 2
forward 2
f_send 2
f_recv 3
forward 3
f_send 3
b_recv 0
backward 0
b_send 0
f_recv 4
forward 4
f_send 4
b_recv 1
backward 1
b_send 1
f_recv 5
forward 5
f_send 5
b_recv 2
backward 2
b_send 2
f_recv 6
forward 6
f_send 6
b_recv 3
backward 3
b_send 3
f_recv 7
forward 7
f_send 7
b_recv 4
backward 4
b_send 4
b_recv 5
backward 5
b_send 5
b_recv 6
backward 6
b_send 6
b_recv 7
backward 7
b_send 7


In [19]:
class Dd:
    def __init__(self):
        self.x = 0
    def __iter__(self):
        self.x = 0
        return self

    def __next__(self):
        if self.x <= 3:
            raise StopIteration
        val = self.x
        self.x += 1
        return val
    



d = Dd()

In [20]:
for i in d:
    print(i)

In [17]:
d[0]

TypeError: 'Dd' object is not subscriptable